# Step 6 — Observation and Prediction Window Design

This analysis evaluates alternative temporal prediction designs for
clinical deterioration in the MIMIC-IV Demo dataset.

The purpose is to identify a clinically interpretable prediction design
that preserves temporal separation between predictor measurements and
future deterioration events while retaining sufficient positive events
for proof-of-concept machine-learning analysis.

Candidate designs are compared using the number of eligible ICU stays,
positive deterioration events, negative stays, event rate, and event
type composition.

In [ ]:

# ============================================================
# STEP 6 — OBSERVATION AND PREDICTION WINDOW DESIGN
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 6.1 LOAD STEP 5 OUTCOME DATA
# ============================================================

outcomes = pd.read_csv(
    "../results/deterioration_outcomes.csv"
)

# Convert timestamp columns back to datetime
time_columns = [
    "intime",
    "outtime",
    "event_time"
]

for col in time_columns:
    outcomes[col] = pd.to_datetime(
        outcomes[col],
        errors="coerce"
    )


# Basic structural checks
print("=" * 60)
print("STEP 6 — PREDICTION WINDOW DESIGN")
print("=" * 60)

print("\nDataset check")
print("-------------------------")
print("Rows:", len(outcomes))
print(
    "Unique ICU stays:",
    outcomes["stay_id"].nunique()
)
print(
    "Unique patients:",
    outcomes["subject_id"].nunique()
)
print(
    "Duplicate ICU stays:",
    outcomes["stay_id"].duplicated().sum()
)


# ============================================================
# 6.2 DEFINE CANDIDATE TEMPORAL DESIGNS
# ============================================================

candidate_designs = [
    {
        "design": "2h → 8h",
        "observation_hours": 2,
        "prediction_end_hours": 8
    },
    {
        "design": "4h → 12h",
        "observation_hours": 4,
        "prediction_end_hours": 12
    },
    {
        "design": "6h → 12h",
        "observation_hours": 6,
        "prediction_end_hours": 12
    },
    {
        "design": "6h → 24h",
        "observation_hours": 6,
        "prediction_end_hours": 24
    },
    {
        "design": "12h → 24h",
        "observation_hours": 12,
        "prediction_end_hours": 24
    }
]


# ============================================================
# 6.3 FUNCTION TO CREATE ELIGIBLE DATASET
# ============================================================

def get_eligible_dataset(
    df,
    observation_hours,
    prediction_end_hours
):
    """
    Create the eligible ICU cohort for a landmark prediction design.

    Patients must:
    1. Still be in the ICU at the landmark.
    2. Have no deterioration event before or at the landmark.

    The outcome is deterioration after the landmark and before
    or at the end of the prediction window.
    """

    temp = df.copy()

    # End of observation period / prediction landmark
    temp["landmark_time"] = (
        temp["intime"]
        + pd.to_timedelta(
            observation_hours,
            unit="h"
        )
    )

    # End of prediction period
    temp["prediction_end_time"] = (
        temp["intime"]
        + pd.to_timedelta(
            prediction_end_hours,
            unit="h"
        )
    )

    # Patient must still be in ICU at landmark
    temp["reaches_landmark"] = (
        temp["outtime"]
        >= temp["landmark_time"]
    )

    # Determine whether deterioration already occurred
    temp["event_before_or_at_landmark"] = (
        temp["event_time"].notna()
        & (
            temp["event_time"]
            <= temp["landmark_time"]
        )
    )

    # Eligible patients are still in ICU and event-free
    temp["eligible"] = (
        temp["reaches_landmark"]
        & ~temp["event_before_or_at_landmark"]
    )

    # Future deterioration target
    temp["future_deterioration"] = (
        temp["eligible"]
        & temp["event_time"].notna()
        & (
            temp["event_time"]
            > temp["landmark_time"]
        )
        & (
            temp["event_time"]
            <= temp["prediction_end_time"]
        )
    ).astype(int)

    # Keep only eligible ICU stays
    eligible = temp.loc[
        temp["eligible"]
    ].copy()

    return eligible


# ============================================================
# 6.4 COMPARE ALL CANDIDATE DESIGNS
# ============================================================

results = []

for design in candidate_designs:

    temp = get_eligible_dataset(
        outcomes,
        observation_hours=design[
            "observation_hours"
        ],
        prediction_end_hours=design[
            "prediction_end_hours"
        ]
    )

    total_eligible = len(temp)

    unique_patients = (
        temp["subject_id"].nunique()
    )

    positives = int(
        temp["future_deterioration"].sum()
    )

    negatives = (
        total_eligible - positives
    )

    event_rate = (
        positives / total_eligible * 100
        if total_eligible > 0
        else np.nan
    )

    results.append(
        {
            "design": design["design"],
            "observation_hours":
                design["observation_hours"],
            "prediction_end_hours":
                design["prediction_end_hours"],
            "eligible_stays":
                total_eligible,
            "unique_patients":
                unique_patients,
            "positive_events":
                positives,
            "negative_stays":
                negatives,
            "event_rate_percent":
                event_rate
        }
    )


design_comparison = pd.DataFrame(
    results
)


# Round event rate for easier reading
design_comparison[
    "event_rate_percent"
] = design_comparison[
    "event_rate_percent"
].round(2)


print("\n")
print("=" * 60)
print("CANDIDATE DESIGN COMPARISON")
print("=" * 60)

display(design_comparison)

# Required summary for the 2h -> 8h design
summary_2h_8h = get_eligible_dataset(
    outcomes,
    observation_hours=2,
    prediction_end_hours=8
)

print("\nEligible ICU stays:", len(summary_2h_8h))
print("Unique patients:", summary_2h_8h["subject_id"].nunique())
print("\nOutcome:")
print(
    summary_2h_8h["future_deterioration"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ============================================================
# 6.5 EXAMINE EVENT TYPE COMPOSITION
# ============================================================

print("\n")
print("=" * 60)
print("EVENT TYPE COMPOSITION")
print("=" * 60)

event_type_results = []


for design in candidate_designs:

    temp = get_eligible_dataset(
        outcomes,
        observation_hours=design[
            "observation_hours"
        ],
        prediction_end_hours=design[
            "prediction_end_hours"
        ]
    )

    positive = temp.loc[
        temp["future_deterioration"] == 1
    ].copy()

    print(
        f"\n{design['design']}"
    )

    print("-" * 40)

    print(
        "Eligible ICU stays:",
        len(temp)
    )

    print(
        "Unique patients:",
        temp["subject_id"].nunique()
    )

    print(
        "Positive events:",
        len(positive)
    )

    print("Event types:")

    if len(positive) > 0:

        event_counts = (
            positive["event_type"]
            .value_counts()
        )

        print(event_counts)

        for event_type, count in event_counts.items():

            event_type_results.append(
                {
                    "design":
                        design["design"],
                    "event_type":
                        event_type,
                    "count":
                        count
                }
            )

    else:

        print(
            "No positive events"
        )


# ============================================================
# 6.6 CREATE EVENT COMPOSITION TABLE
# ============================================================

event_type_comparison = pd.DataFrame(
    event_type_results
)

if len(event_type_comparison) > 0:

    event_type_table = (
        event_type_comparison
        .pivot_table(
            index="design",
            columns="event_type",
            values="count",
            fill_value=0
        )
        .reset_index()
    )

    print("\n")
    print("=" * 60)
    print("EVENT TYPE COMPARISON TABLE")
    print("=" * 60)

    display(event_type_table)

else:

    event_type_table = pd.DataFrame()

    print(
        "\nNo positive events were found "
        "in any candidate design."
    )


# ============================================================
# 6.7 SANITY CHECK — REPRODUCE STEP 5 RESULT
# ============================================================

check_6_12 = get_eligible_dataset(
    outcomes,
    observation_hours=6,
    prediction_end_hours=12
)

print("\n")
print("=" * 60)
print("6h → 12h SANITY CHECK")
print("=" * 60)

print(
    "Eligible at 6-hour landmark:",
    len(check_6_12)
)

print(
    "Deterioration during 6–12 h:",
    int(
        check_6_12[
            "future_deterioration"
        ].sum()
    )
)

print(
    "No deterioration during 6–12 h:",
    int(
        (
            check_6_12[
                "future_deterioration"
            ] == 0
        ).sum()
    )
)


# ============================================================
# 6.8 SAVE RESULTS
# ============================================================

design_comparison.to_csv(
    "../results/prediction_window_comparison.csv",
    index=False
)

if len(event_type_table) > 0:

    event_type_table.to_csv(
        "../results/prediction_window_event_types.csv",
        index=False
    )


print("\n")
print("=" * 60)
print("STEP 6 ANALYSIS COMPLETE")
print("=" * 60)

print(
    "Saved: "
    "../results/prediction_window_comparison.csv"
)

if len(event_type_table) > 0:

    print(
        "Saved: "
        "../results/prediction_window_event_types.csv"
    )

STEP 6 — PREDICTION WINDOW DESIGN

Dataset check
-------------------------
Rows: 140
Unique ICU stays: 140
Unique patients: 100
Duplicate ICU stays: 0


CANDIDATE DESIGN COMPARISON


,design,observation_hours,prediction_end_hours,eligible_stays,unique_patients,positive_events,negative_stays,event_rate_percent
0,2h → 8h,2,8,99,79,18,81,18.18
1,4h → 12h,4,12,86,68,8,78,9.30
2,6h → 12h,6,12,80,63,2,78,2.50
3,6h → 24h,6,24,80,63,8,72,10.00
4,12h → 24h,12,24,76,60,6,70,7.89




EVENT TYPE COMPOSITION

2h → 8h
----------------------------------------
Eligible ICU stays: 99
Unique patients: 79
Positive events: 18
Event types:
event_type
Mechanical ventilation    18
Name: count, dtype: int64

4h → 12h
----------------------------------------
Eligible ICU stays: 86
Unique patients: 68
Positive events: 8
Event types:
event_type
Mechanical ventilation    7
Vasopressor               1
Name: count, dtype: int64

6h → 12h
----------------------------------------
Eligible ICU stays: 80
Unique patients: 63
Positive events: 2
Event types:
event_type
Vasopressor               1
Mechanical ventilation    1
Name: count, dtype: int64

6h → 24h
----------------------------------------
Eligible ICU stays: 80
Unique patients: 63
Positive events: 8
Event types:
event_type
Vasopressor               4
Mechanical ventilation    4
Name: count, dtype: int64

12h → 24h
----------------------------------------
Eligible ICU stays: 76
Unique patients: 60
Positive events: 6
Event types:

event_type,design,Mechanical ventilation,Vasopressor
0,12h → 24h,3.0,3.0
1,2h → 8h,18.0,0.0
2,4h → 12h,7.0,1.0
3,6h → 12h,1.0,1.0
4,6h → 24h,4.0,4.0




6h → 12h SANITY CHECK
Eligible at 6-hour landmark: 80
Deterioration during 6–12 h: 2
No deterioration during 6–12 h: 78


STEP 6 ANALYSIS COMPLETE
Saved: ../results/prediction_window_comparison.csv
Saved: ../results/prediction_window_event_types.csv
